# Data engineering with Databricks - Building our Manufacturing IOT platform

Building an IOT platform requires to ingest multiple datasources.  

It's a complex process requiring batch loads and streaming ingestion to support real-time insights, used for real-time monitoring among others.

Ingesting, transforming and cleaning data to create clean SQL tables for our downstream user (Data Analysts and Data Scientists) is complex.

<link href="https://fonts.googleapis.com/css?family=DM Sans" rel="stylesheet"/>
<div style="width:300px; text-align: center; float: right; margin: 30px 60px 10px 10px;  font-family: 'DM Sans'">
  <div style="height: 300px; width: 300px;  display: table-cell; vertical-align: middle; border-radius: 50%; border: 25px solid #fcba33ff;">
    <div style="font-size: 70px;  color: #70c4ab; font-weight: bold">
      73%
    </div>
    <div style="color: #1b5162;padding: 0px 30px 0px 30px;">of enterprise data goes unused for analytics and decision making</div>
  </div>
  <div style="color: #bfbfbf; padding-top: 5px">Source: Forrester</div>
</div>

<br>

## <img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/john.png" style="float:left; margin: -35px 0px 0px 0px" width="80px"> John, as Data engineer, spends immense time….


* Hand-coding data ingestion & transformations and dealing with technical challenges:<br>
  *Supporting streaming and batch, handling concurrent operations, small files issues, GDPR requirements, complex DAG dependencies...*<br><br>
* Building custom frameworks to enforce quality and tests<br><br>
* Building and maintaining scalable infrastructure, with observability and monitoring<br><br>
* Managing incompatible governance models from different systems
<br style="clear: both">

This results in **operational complexity** and overhead, requiring expert profile and ultimatly **putting data projects at risk**.


<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F01-Data-ingestion%2F01.2-SDP-python%2F01.1-SDP-Wind-Turbine-Python&demo_name=lakehouse-iot-platform&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-iot-platform%2F01-Data-ingestion%2F01.2-SDP-python%2F01.1-SDP-Wind-Turbine-Python&version=1">

# Simplify Ingestion and Transformation with Spark Declarative Pipelines

<img style="float: right" width="500px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/team_flow_john.png" />

In this notebook, we'll work as a Data Engineer to build our IOT platform. <br>
We'll ingest and clean our raw data sources to prepare the tables required for our BI & ML workload.

Databricks simplifies this task with Spark Declarative Pipelines by making Data Engineering accessible to all.

Spark Declarative Pipelines allows Data Analysts to create advanced pipeline with plain SQL, or python.

Your Spark Declarative Pipeline has been installed and started for you! Open the <a dbdemos-pipeline-id="sdp-sql" href="#joblist/pipelines/0dff17da-ec3c-4b96-8358-efe337f3c50c" target="_blank">IOT Wind Turbine Spark Declarative Pipeline</a> to see it in action.<br/>

*(Note: The pipeline will automatically start once the initialization job is completed with dbdemos, this might take a few minutes... Check installation logs for more details)*

## Building a Spark Declarative Pipeline to ingest IOT sensor and detect faulty equipments

In this example, we'll implement a end 2 end Spark Declarative Pipeline consuming our Wind Turbine sensor data. <br/>
We'll use the medaillon architecture but we could build star schema, data vault or any other modelisation.

We'll incrementally load new data with the autoloader, enrich this information and then load a model from MLFlow to perform our predictive maintenance analysis.

This information will then be used to build our AI/BI dashboards to track our wind turbine farm status, faulty equipment impact and recommendations to reduce potential downtime.

### Dataset:

* <strong>Turbine metadata</strong>: Turbine ID, location (1 row per turbine)
* <strong>Turbine sensor stream</strong>: Realtime streaming flow from wind turbine sensor (vibration, energy produced, speed etc)
* <strong>Turbine status</strong>: Historical turbine status based to analyse which part is faulty (used as label in our ML model)


Let's implement the following flow: 
 
<div><img width="1100px" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-full.png"/></div>

*Note that we're including the ML model our [Data Scientist built]($../04-Data-Science-ML/04.1-automl-predictive-maintenance-turbine) using Databricks AutoML to predict the churn. We'll cover that in the next section.*


## 1/ Data Exploration

All Data projects start with some exploration. Open the [/explorations/sample_exploration]($./explorations/sample_exploration) notebook to get started and discover the data made available to you


## 2/ Ingest data: Bronze layer

<div><img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-1.png" width="700px" style="float: right"/></div>

Ingesting data from stream source can be challenging. In this example we'll incrementally load the files from our cloud storage, only getting the new one (in near real-time or triggered every X hours).

Note that while our streaming data is added to our cloud storage, we could easily ingest from kafka directly : `.format(kafka)`

Auto-loader provides for you:

- Schema inference and evolution
- Scalability handling million of files
- Simplicity: just define your ingestion folder, Databricks take care of the rest!

For more details on autoloader, run `dbdemos.install('data-ingestion')`

Let's use it to our pipeline and ingest the raw JSON & CSV data being delivered in our blob storage `/demos/manufacturing/iot_turbine/...`. 

Open the [transformations/01-bronze.sql]($./transformations/01-bronze.sql) notebook to review the SQL queries ingesting the raw data and creating our bronze layer.


## 3/ Silver layer

<div><img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-2.png" width="700px" style="float: right"/></div>

To be able to analyze our data, we'll compute statistical metrics every at an ourly basis, such as standard deviation and quartiles.

*Note that we'll be recomputing all the table to keep this example simple. We could instead UPSERT the current hour with a stateful agregation*

Open the [transformations/02-silver.sql]($./transformations/02-silver.sql) notebook to review the SQL queries creating our features and our training dataset


## 4/ Gold layer: Get model from registry and add flag faulty turbines

<div><img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-4.png" width="700px" style="float: right"/></div>

Our Data scientist team has been able to read data from our previous table and build a predictive maintenance model using Auto ML and saved it into Databricks Model registry (we'll see how to do that next).

One of the key value of the Lakehouse is that we can easily load this model and predict faulty turbines with into our pipeline directly.

Note that we don't have to worry about the model framework (sklearn or other), MLFlow abstract that for us.

All we have to do is load the model, and call it as a SQL function (or python)

Open the [transformations/03-gold.sql]($./transformations/03-gold.sql) notebook to review the SQL queries creating our features and our training dataset


## Conclusion
Our <a dbdemos-pipeline-id="sdp-sql" href="#joblist/pipelines/0dff17da-ec3c-4b96-8358-efe337f3c50c" target="_blank">Spark Declarative Pipeline</a> is now ready using purely SQL. We have an end 2 end cycle, and our ML model has been integrated seamlessly by our Data Engineering team.


For more details on model training, open the [model training notebook]($../../04-Data-Science-ML/04.1-automl-iot-turbine-predictive-maintenance)

Our final dataset includes our ML prediction for our Predictive Maintenance use-case. 

We are now ready to build our <a dbdemos-dashboard-id="turbine-analysis" href="/sql/dashboardsv3/01f1b2d439eb1ceb9defd16408db9be1">AI/BI Dashboard</a> to track the main KPIs and status of our entire Wind Turbine Farm and build complete <a dbdemos-dashboard-id="turbine-predictive" href="/sql/dashboardsv3/01f1b2d439eb1ceb9defd16408db9be1">Predictive maintenance AI/BI Dashboard</a>.


<img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-dashboard-1.png" width="1000px">